## Reading Bronze.Aircrafts Delta Table

In [0]:
Aircrafts_bronze_path = "s3://travel-analytics-bronze/delta/bronze/aircrafts/"
Aircrafts_bronze_df = spark.read.format("delta").load(Aircrafts_bronze_path)

## Silver Transformations

In [0]:
#importing the needed libraries
from pyspark.sql import functions as F
from pyspark.sql.functions import (
col, trim, upper, to_date, to_timestamp,
    when, date_format, concat, lit, coalesce, expr, initcap
)
# =============================================================
# STEP 0:CONFIGURATION & SETUP
# =============================================================
table_name = "aircrafts"
aircraft_silver_path = f"s3://travel-analytics-bronze/delta/silver/{table_name}/"

# =============================================================
# STEP 1: DATA TYPE CASTING & Parsing
# =============================================================
print("\nSTEP 1: Casting Data Types (Aircraft)...")

aircraft_step_1_df = (
    Aircrafts_bronze_df

    # ========== String columns ==========
    .withColumn("aircraft_id", trim(col("aircraft_id")))
    .withColumn("aircraft_iata", trim(upper(col("aircraft_iata"))))
    .withColumn("aircraft_icao", trim(upper(col("aircraft_icao"))))
    .withColumn("aircraft_name", initcap(trim(col("aircraft_name"))))  # First char capital

    # ========== CDC updated timestamp ==========
    .withColumn("updated_at", to_timestamp(col("_ab_cdc_updated_at")))
)

# ===================================================================================
# STEP 2: Cleaning , Standardizing String Columns and  Business Logic applications
# ===================================================================================
print("\nSTEP 2: Cleaning & Standardization (Aircraft)...")

aircraft_step_2_df = (
    aircraft_step_1_df

    # Handle nulls for string columns
    .withColumn("aircraft_name", coalesce(col("aircraft_name"), lit("UNKNOWN")))
    .withColumn("aircraft_iata", coalesce(col("aircraft_iata"), lit("UNKNOWN")))
    .withColumn("aircraft_icao", coalesce(col("aircraft_icao"), lit("UNKNOWN")))
)
# ===================================================================================
#  Step 3: Deduplicating Using Composite PK("customer_id", "hotel_id", "check_in_date")
#          Dropping unwanted Airbyte metadata columns
# ===================================================================================
print("\nSTEP 3: Deduplication & Dropping Airbyte Columns (Aircraft)...")

airbyte_columns_to_drop = [
    "_airbyte_ab_id",
    "_airbyte_emitted_at",
    "_airbyte_additional_properties",
    "_ab_cdc_lsn",
    "_ab_cdc_deleted_at",
]

aircraft_silver_df = (
    aircraft_step_2_df

    # Deduplicate on natural key
    .dropDuplicates(["aircraft_id"])

    # Drop metadata
    .drop(*airbyte_columns_to_drop)
)


STEP 1: Casting Data Types (Aircraft)...

STEP 2: Cleaning & Standardization (Aircraft)...

STEP 3: Deduplication & Dropping Airbyte Columns (Aircraft)...


In [0]:
#=============================================================
# STEP 4: BUSINESS-FRIENDLY COLUMN NAMES
#=============================================================

print("\nSTEP 4: Renaming Columns (Aircraft)...")

rename_map = {
    "aircraft_id": "Aircraft_Id",
    "aircraft_iata": "Aircraft_IATA",
    "aircraft_icao": "Aircraft_ICAO",
    "aircraft_name": "Aircraft_Name",
    "updated_at": "Updated_At"
}

aircraft_silver_df = aircraft_silver_df.select(
    [col(c).alias(rename_map.get(c, c)) for c in aircraft_silver_df.columns]
)



STEP 4: Renaming Columns (Aircraft)...


In [0]:
aircraft_silver_df.display()

Aircraft_Id,Aircraft_IATA,Aircraft_ICAO,Aircraft_Name,_ab_cdc_updated_at,Updated_At
100F100,100,F100,Fokker 100,2025-12-12T01:04:52.921893877Z,2025-12-12T01:04:52.921Z
141B461,141,B461,Bae 146-100,2025-12-12T01:04:52.921893877Z,2025-12-12T01:04:52.921Z
142B462,142,B462,Bae 146-200,2025-12-12T01:04:52.921893877Z,2025-12-12T01:04:52.921Z
143B463,143,B463,Bae 146-300,2025-12-12T01:04:52.921893877Z,2025-12-12T01:04:52.921Z
146,146,UNKNOWN,Bae 146,2025-12-12T01:04:52.921893877Z,2025-12-12T01:04:52.921Z
290E290,290,E290,Embraer E190-e2,2025-12-12T01:04:52.921893877Z,2025-12-12T01:04:52.921Z
295E295,295,E295,Embraer E195-e2,2025-12-12T01:04:52.921893877Z,2025-12-12T01:04:52.921Z
310A310,310,A310,Airbus A310,2025-12-12T01:04:52.921893877Z,2025-12-12T01:04:52.921Z
318A318,318,A318,Airbus A318,2025-12-12T01:04:52.921893877Z,2025-12-12T01:04:52.921Z
319A319,319,A319,Airbus A319,2025-12-12T01:04:52.921893877Z,2025-12-12T01:04:52.921Z


## Writing Silver Aircrafts to Delta Lake

In [0]:
print("\nSTEP 5: Persist Aircraft Silver Table...")

aircraft_silver_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save(aircraft_silver_path)


STEP 5: Persist Aircraft Silver Table...
